# Azimuthal Averaging

This guide showcases how to apply Azimuthal Averaging on 2D and 3D fields using UXarray.

## Azimuthal mean basics
An azimuthal average (or azimuthal mean) is a statistical measure that represents the average of a face-centered variable along rings/bands of constant distance from a specified central point. Azimuthal averaging is useful for describing circular/cylindrical features, where fields strongly depend on distance from a point.

In UXarray, azimuthal averaging is non-conservative. This means that faces are assigned to radial bins (i.e., distance intervals from the central point) based only on the face center coordinate.

In [ ]:
# The standard library imports and cartopy are used to visualize range rings in this demo
# and are not needed for routine use of `azimuthal_mean()`
import math
import operator
from functools import reduce

import cartopy.geodesic as gdyn
import holoviews as hv
import matplotlib.pyplot as plt
import numpy as np

import uxarray as ux

In [ ]:
uxds = ux.tutorial.open_dataset("outCSne30-vortex")

## 1. Azimuthally averaging an idealized 2D field

### Helper function to draw range rings and visualize azimuthal averaging:

In [ ]:
def hv_range_ring(lon, lat, rad_gcd, n_samples=2000, color="red", line_width=1):
    geo = gdyn.Geodesic()
    circ_pts = geo.circle(
        lon=lon, lat=lat, radius=rad_gcd * 111320, n_samples=n_samples
    )
    return hv.Path(circ_pts).opts(color=color, line_width=line_width)

### Step 1.1: Visualize the global field
The global field is shaded below using the UxDataArray.plot() accessor. To help visualize azimuthal averaging, the chosen central point is marked with an 'x' and rings are drawn at every 10 great-circle degrees from the central point (37°N, 1°E).

In [ ]:
# Display the global field
glob_plt = uxds["psi"].plot(
    cmap="inferno", periodic_elements="split", title="Global Field", dynamic=True
)

# Mark a center coordinate and draw range rings
lon, lat = 1, 37
glob_plt = glob_plt * hv.Points([(lon, lat)]).opts(
    color="lime", marker="x", size=10, line_width=2
)
glob_plt = glob_plt * reduce(
    operator.mul,
    [hv_range_ring(lon, lat, rr, color="lime") for rr in np.arange(10, 41, 10)],
)
glob_plt

### Step 1.2: Compute the azimuthal mean
Calling `.azimuthal_mean()` with the arguments below samples every 2° out to 40 great-circle degrees around the central point.

In [ ]:
azim_mean_psi, hits = uxds["psi"].azimuthal_mean(
    (lon, lat), 40.0, 2, return_hit_counts=True
)
azim_mean_psi

`azimuthal_mean()` can also be called directly on a UxDataset to compute the zonal mean for every face-centered variable (the example dataset here has only 1 variable, but this can be especially convenient for a dataset with many variables), although it doesn't yet support the `return_hit_counts` parameter:

In [ ]:
azim_mean_uxds = uxds.azimuthal_mean((lon, lat), 40.0, 2)
print(
    "Equivalent to UxDataArray.azimuthal_mean:",
    azim_mean_uxds["psi"].equals(azim_mean_psi),
)
azim_mean_uxds

### Step 1.3: Plot the azimuthal mean
In the plot below, red dots mark where samples were taken at 2° intervals. The green vertical lines correspond to the green rings in the global field plot.

In [ ]:
plt.plot(azim_mean_psi["radius"], azim_mean_psi)
plt.scatter(azim_mean_psi["radius"], azim_mean_psi, s=10, color="red")
plt.xlabel("radius [great-circle degrees]")
plt.ylabel("psi")

[plt.axvline(rr, color="lime") for rr in np.arange(10, 41, 10)]

### Step 1.4: Inspect the hit count
The plot below shows the number of face centers that fall within each distance bin. As one would expect on a near-uniformly spaced mesh, the hit count increases linearly with distance.

In [ ]:
plt.plot(azim_mean_psi["radius"], hits)
plt.scatter(azim_mean_psi["radius"], hits, s=10, color="red")

## 2: Azimuthally averaging tropical cyclone fields
A high-resolution (~0.25°) aquaplanet general circulation model permits the development of a handful of strong tropical cyclones (TCs). Because TCs are fairly axisymmetric, azimuthal averaging is useful for transforming their fields into cylindrical coordinates and visualizing features such as their low central pressure and warm core.

In [ ]:
clon, clat = 114.54, -17.66
tcds = ux.tutorial.open_dataset("ne120-tcsubset").squeeze()

### Step 2.1: Visualize the raw surface pressure field
Because we are only interested in a single tropical cyclone, the demo dataset has already been subsetted. Otherwise, you can use UXArray subsetting operations to select a region of the global field and speed up plotting.

In [ ]:
ps_plt = tcds["PS"].plot(
    cmap="inferno", periodic_elements="split", title="Surface pressure", dynamic=True
)
ps_plt

### Step 2.2: Compute the azimuthal mean of the 3D fields

In [ ]:
args = ((clon, clat), 3, 0.25)
azim_mean_T = tcds["T"].azimuthal_mean(*args)
azim_mean_Z = tcds["Z3"].azimuthal_mean(*args)
azim_mean_T

### Step 2.3: Plot the TC radial profile
The contour plot below shows the warm core and low pressure of the TC. After taking the azimuthal average, we subtract the value at the outermost radius to obtain an approximate anomaly relative to the ambient environment.

In [ ]:
# Subtract the value at the outer radius
Tpert = azim_mean_T - azim_mean_T.isel(radius=-1)
Zpert = azim_mean_Z - azim_mean_Z.isel(radius=-1)

plt.contour(
    azim_mean_Z["radius"],
    tcds["plev"] / 100,
    Zpert,
    levels=np.arange(-500, 501, 50),
    colors="black",
)
plt.contourf(
    azim_mean_T["radius"],
    tcds["plev"] / 100,
    Tpert,
    levels=np.arange(-10, 11, 2),
    cmap="bwr",
    extend="both",
)
plt.xlabel("distance from center [°]")
plt.ylim(1000, 100)
plt.yscale("log")
plt.ylabel("Pressure [hPa]")
plt.colorbar(label="T anomaly [K]")

## 3. Chunked (lazy) azimuthal means
Opening a dataset with `chunks` backs its variables with Dask, and `azimuthal_mean()` then returns a lazy result. The ring that each face falls in is found from the grid right away, but no data is read or averaged until the result is materialized with `.compute()`, etc.

### Step 3.1: Build the lazy azimuthal mean
Below, the tropical cyclone fields are split into two chunks of 13 pressure levels and into chunks of 500 faces.

In [ ]:
tcds_chunked = ux.tutorial.open_dataset(
    "ne120-tcsubset", chunks={"plev": 13, "n_face": 500}
).squeeze()

azim_mean_T_lazy, hits_T = tcds_chunked["T"].azimuthal_mean(
    *args, return_hit_counts=True
)
azim_mean_T_lazy

The result keeps the input's `plev` chunks and holds every radius in a single chunk. The hit counts depend only on the grid, so they are returned as a NumPy-backed array.

### Step 3.2: Compute the result
Computing the lazy mean gives the same profile as the in-memory calculation in Step 2.2.

In [ ]:
azim_mean_T_lazy = azim_mean_T_lazy.compute()
np.testing.assert_allclose(azim_mean_T_lazy, azim_mean_T)

### Step 3.3: Estimate the memory of a chunk
All ring means come from one sparse weighted sum over every face within `outer_radius`, so those faces are gathered into a single `n_face` chunk. Each task holds all of them for one chunk of the other dimensions:

In [ ]:
n_ring_faces = int(hits_T.sum())
plev_chunk = max(tcds_chunked["T"].chunksizes["plev"])
block_bytes = n_ring_faces * plev_chunk * tcds_chunked["T"].dtype.itemsize
print(
    f"{n_ring_faces} faces x {plev_chunk} levels per chunk = {block_bytes / 1e3:.1f} kB"
)

### Notes on lazy execution and memory
- **Dask-backed data** (opened with `chunks=`, or after `.chunk()`): `azimuthal_mean()` builds a task graph and returns immediately. When it is computed, only the input `n_face` chunks that contain a face within `outer_radius` are read, and each of those is read in full, so smaller `n_face` chunks read less data around a small ring.
- **Lazily loaded data without Dask** (the default for `ux.open_dataset()`): the faces within `outer_radius` are read when `azimuthal_mean()` is called, for every time step and level. They are read as a few contiguous slices that cover at most twice as many faces as were selected, and the result is held in memory.
- **In-memory data**: the result is computed immediately.
- The grid's face centers and face areas are always loaded to find the rings; only the data variable is lazy.
- **Single face chunk**: the faces within `outer_radius` form one `n_face` chunk, so each task needs memory for those faces times one chunk of the other dimensions (Step 3.3), plus the input chunks it gathers them from and the float64 result. The input's `n_face` chunks only control what is read from disk. To limit memory for a large `outer_radius` or a high-resolution grid, chunk the other dimensions (such as `time` or `lev`) more finely; smaller `n_face` chunks do not reduce it.

### Numerical and boundary behavior
- Faces are binned by the straight-line (chord) distance $d$ from the center point to their face center, compared in float64: a face is in the ring at radius $r_i$ when $r_{i-1} < d \le r_i$. A face center that lies on a ring boundary counts toward the inner ring. Earlier releases found each ring with a separate kd-tree query, whose rounding could put such a face in the neighboring ring or leave it out at `outer_radius`, so hit counts and means can differ slightly for rings with faces on their boundaries.
- Away from ring boundaries, the means agree with earlier releases to floating-point rounding (about $10^{-15}$ relative), since each ring's weighted sum is now added up in a different order. Chunked and in-memory inputs use the same sums, and the result is float64, including for float32 data.
- The ring at radius 0 only holds a face whose center coincides with the center point, so it is usually empty. Empty rings, and radii past 180°, where the rings shrink back toward the antipode, have a hit count of 0 and a mean of NaN.
- NaN values add nothing to a ring's weighted sum, but their face areas still count toward the ring's area. Missing values therefore pull a ring's mean toward 0, and a ring with only NaN values has a mean of 0. To average only the valid faces, divide by the mean of a validity mask:

```python
valid = ux.UxDataArray(field.notnull(), uxgrid=field.uxgrid)
nan_aware = field.fillna(0).azimuthal_mean(*args) / valid.azimuthal_mean(*args)
```